# Zonas verdes por distrito

Objetivo: calcular cuántos m² de espacios verdes hay en cada distrito de Valencia.

Datos:
- `districtes-distritos.geojson`: los distritos.
- `espais-verds-espacios-verdes.geojson`: el inventario de espacios verdes del Ayuntamiento.

Como algunos parques están en dos distritos (por ejemplo, el antiguo cauce del Turia), se recorta cada parque por los límites de los distritos y se suma la superficie que cae en cada uno.

## 1. Librerías

In [ ]:
# pip install geopandas
import geopandas as gpd

## 2. Leer los archivos

In [ ]:
distritos = gpd.read_file("districtes-distritos.geojson")
verdes = gpd.read_file("espais-verds-espacios-verdes.geojson")

print("Distritos:", len(distritos))
print("Espacios verdes:", len(verdes))

## 3. Preparar los datos

- Me quedo solo con las columnas que necesito.
- Paso a EPSG:25830 para que las áreas salgan en metros cuadrados (el original está en grados).
- `buffer(0)` arregla algunos polígonos que vienen mal dibujados.

In [ ]:
distritos = distritos[["coddistrit", "nombre", "geometry"]]
distritos = distritos.rename(columns={"nombre": "distrito"})
distritos["coddistrit"] = distritos["coddistrit"].astype(int)
distritos = distritos.to_crs(25830)

verdes = verdes[["id_jardin", "nombre", "tipologia", "geometry"]]
verdes = verdes.to_crs(25830)
verdes["geometry"] = verdes.buffer(0)

print("Polígonos no válidos:", (~verdes.is_valid).sum())

## 4. Recortar los espacios verdes por distrito

`overlay` con `intersection` corta cada parque por los límites de los distritos. Si un parque está en dos distritos, salen dos trozos.

In [ ]:
trozos = gpd.overlay(verdes, distritos, how="intersection")
trozos["m2"] = trozos.area

print("Trozos:", len(trozos))
trozos.head()

## 5. Sumar por distrito

In [ ]:
resumen = trozos.groupby(["coddistrit", "distrito"]).agg(
    m2_verde=("m2", "sum"),
    n_espacios=("id_jardin", "nunique"),
).reset_index()

resumen["m2_verde"] = resumen["m2_verde"].round(0)
resumen = resumen.sort_values("coddistrit")
resumen

## 6. m² por tipo de espacio verde

Una columna por cada tipología.

In [ ]:
por_tipo = trozos.pivot_table(
    index="coddistrit",
    columns="tipologia",
    values="m2",
    aggfunc="sum",
    fill_value=0,
).round(0)

por_tipo = por_tipo.rename(columns={
    "Parques Urbanos": "m2_parques_urbanos",
    "Jardines Barrio Plaza": "m2_jardines_barrio",
    "Jardines Especial Protección": "m2_jardines_proteccion",
    "Acompañamiento Viario": "m2_viario",
    "Bulevar": "m2_bulevar",
})

resumen = resumen.merge(por_tipo, on="coddistrit", how="left")
resumen

## 7. Comprobaciones

- Tienen que salir 19 distritos.
- La suma por distritos tiene que ser casi igual a la superficie total de los espacios verdes.

In [ ]:
print("Distritos en el resultado:", len(resumen))

total_original = verdes.area.sum()
total_resumen = resumen["m2_verde"].sum()
print("m² totales originales:", round(total_original))
print("m² totales por distritos:", round(total_resumen))
print("Diferencia (%):", round((total_original - total_resumen) / total_original * 100, 2))

## 8. Guardar el resultado

In [ ]:
resumen = resumen.rename(columns={"coddistrit": "id_distrito"})
resumen.to_csv("verde_distrito.csv", index=False)
print("Guardado verde_distrito.csv")